# Cost and Latency Engineering for AI Systems at Volume

## What you will build

You will build the model calls behind an insurance claims desk. Adjusters ask live questions about
claims and wait on screen for the answer, while a backlog of claims is scored overnight when nobody
is waiting at all. Both jobs call the same model, but one needs an answer in seconds and the other
only needs to be cheap.

A desk that treats both jobs the same way gets both of them wrong. The diagram shows the three
mistakes this course stops. An adjuster is left waiting on calls made one after another, a cache
saving goes into the budget and never shows up on the bill, and the backlog is scored at the price
of the live lane.

![What you will build](images/claims-desk-overview.svg)

## Step 0: Set up the client and read the model's prices

Every call in this notebook goes through the repository's own client, which calls the model live
with an API key and uses recorded responses from real runs without one. The prices come from the
provider's own model list, saved in the repository, so no number in this course is typed in by
hand.

In [1]:
import json
import statistics
import time
from concurrent.futures import ThreadPoolExecutor
from types import SimpleNamespace

from vault import Usage, cost_of, get_client, load_env, model_for, provider_truth

load_env()
client = get_client("13-cost-and-latency-at-volume/01-route-claims-by-cost-and-latency")
MODEL = model_for("default")
PRICES = provider_truth()["models"]

print(f"Client ready. The live lane uses {MODEL}.")
print(f"list price per prompt token     : ${PRICES[MODEL]['prompt_usd_per_token']}")
print(f"list price per completion token : ${PRICES[MODEL]['completion_usd_per_token']}")

Client ready. The live lane uses google/gemini-2.5-flash-lite.
list price per prompt token     : $0.0000001
list price per completion token : $0.0000004


## Step 1: Write the claims manual and the two workloads

Every question an adjuster asks is answered from the insurer's claims manual, so the manual goes
into every request. It is long and it never changes between questions, which is exactly the kind
of text a provider may be able to reuse from one call to the next.

In [2]:
PERILS = ["water", "fire", "storm", "theft", "collision", "glass"]
EVIDENCE = ["photos", "a receipt", "a police report", "a repair quote"]
ESCALATE_IF = ["anyone was injured", "there is a prior claim", "notice came late"]

CLAIMS_MANUAL = ("You answer questions from insurance claims adjusters in one short sentence, "
                 "using only these handling rules.\n") + "\n".join(
    f"RULE-{n:03d}: peril {PERILS[n % 6]}, excess {100 + (n % 5) * 50} EUR, "
    f"fast track under {1000 + (n % 7) * 500} EUR, needs {EVIDENCE[n % 4]}, "
    f"escalate if {ESCALATE_IF[n % 3]}"
    for n in range(300))

print(f"claims manual: {len(CLAIMS_MANUAL)} characters, {CLAIMS_MANUAL.count('RULE-')} rules")
print(CLAIMS_MANUAL.splitlines()[15])

claims manual: 35258 characters, 300 rules
RULE-014: peril storm, excess 300 EUR, fast track under 1000 EUR, needs a police report, escalate if notice came late


The two workloads come next. `ADJUSTER_QUESTIONS` are asked live by someone waiting on screen,
and `OVERNIGHT_BACKLOG` is a slice of the claims that get a fraud score before the morning shift.

In [3]:
ADJUSTER_QUESTIONS = [
    "What excess applies under RULE-014?",
    "What evidence does RULE-122 need?",
    "Below what amount is a RULE-087 claim fast tracked?",
    "When must a RULE-203 claim be escalated?",
    "Which peril does RULE-256 cover?",
]
DAMAGE = ["burst pipe under the kitchen sink", "hail cracked the windscreen",
          "bicycle stolen from a locked shed", "pan fire in the kitchen",
          "tree fell on the garage roof", "low speed rear-end collision"]
OVERNIGHT_BACKLOG = [{"claim_id": f"CLM-{4100 + n}", "text": f"{DAMAGE[n % 6]}, {600 + n * 350} EUR"}
                     for n in range(12)]

print(f"{len(ADJUSTER_QUESTIONS)} live questions, {len(OVERNIGHT_BACKLOG)} claims in the backlog")
print(OVERNIGHT_BACKLOG[0])

5 live questions, 12 claims in the backlog
{'claim_id': 'CLM-4100', 'text': 'burst pipe under the kitchen sink, 600 EUR'}


## Step 2: Answer one question and read its usage block

We send one adjuster question and read the usage block that comes back with the answer, because
it is the only record of what the call cost. It counts each **token**, the unit a model reads and
bills in, roughly a short word or part of one, and it says how many of the prompt's tokens came
from a cache.

![Answer one question and read its usage block](images/claims-lanes-step-1.svg)

In [4]:
def build_messages(question):
    """The manual always comes first, byte for byte, so it is the same prefix every time."""
    return [{"role": "system", "content": CLAIMS_MANUAL},
            {"role": "user", "content": question}]


def ask_claims_desk(question, model=MODEL):
    """Ask one question. Return the response and the seconds the call took."""
    started = time.monotonic()
    response = client.chat.completions.create(model=model, max_tokens=80,
                                              messages=build_messages(question))
    return response, time.monotonic() - started

The manual sits at the front of every request as the **prefix**, the unchanging start of a request,
which is the only part a cache can reuse. The question goes after it.

In [5]:
response, FIRST_CALL_SECONDS = ask_claims_desk(ADJUSTER_QUESTIONS[0])
usage = response.usage

print(f"answer                : {response.choices[0].message.content.strip()}")
print(f"prompt_tokens         : {usage.prompt_tokens}")
print(f"completion_tokens     : {usage.completion_tokens}")
print(f"prompt_tokens_details : {usage.prompt_tokens_details}")
print(f"seconds               : {FIRST_CALL_SECONDS:.2f}")

answer                : The excess for RULE-014 is 300 EUR.
prompt_tokens         : 11104
completion_tokens     : 15
prompt_tokens_details : PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0)
seconds               : 0.56


`read_cached_tokens` pulls the cached count out of the usage block. It returns `None` rather than
zero when the block has no cache details at all, because "not reported" and "nothing cached" are
different answers. `price_the_call` puts the list price, worked out from the token counts, next to
the `cost` the provider says it billed.

In [6]:
def read_cached_tokens(usage):
    """Cached prompt tokens, or None when the response does not report them."""
    details = getattr(usage, "prompt_tokens_details", None)
    return getattr(details, "cached_tokens", None) if details else None


def price_the_call(response):
    """The list price from the token counts, and what the provider says it billed."""
    list_usd = cost_of(Usage.from_response(response))
    return list_usd, getattr(response.usage, "cost", None)


def format_usd(amount):
    """Dollars to six places, or 'absent' when the provider did not say."""
    return "absent" if amount is None else f"${amount:.6f}"


list_usd, billed_usd = price_the_call(response)
print(f"cached_tokens : {read_cached_tokens(usage)}")
print(f"list price    : {format_usd(list_usd)}")
print(f"billed        : {format_usd(billed_usd)}")

cached_tokens : 0
list price    : $0.001116
billed        : $0.001116


## Step 3: Watch the cache warm up over repeated questions

**Prompt caching** is the provider reusing work on a repeated prefix, so you pay less for it. The
requests in this notebook ask for nothing special, so whether caching happens is up to the model
and the host that serves it, and the only way to find out is to read `cached_tokens` on every call.

In [7]:
def measure_question(question, model=MODEL):
    """Ask one question and keep the numbers this course compares."""
    response, seconds = ask_claims_desk(question, model)
    list_usd, billed_usd = price_the_call(response)
    return {"prompt": response.usage.prompt_tokens, "cached": read_cached_tokens(response.usage),
            "seconds": seconds, "list_usd": list_usd, "billed_usd": billed_usd,
            "host": getattr(response, "provider", None)}


def print_measurements(calls):
    """One line per call, in the order the calls were made."""
    for n, call in enumerate(calls, start=1):
        print(f"call {n:2}: cached {str(call['cached']):>5} of {call['prompt']}  "
              f"{call['seconds']:4.2f}s  list {format_usd(call['list_usd'])}  "
              f"billed {format_usd(call['billed_usd'])}  host {call['host']}")

The next cell asks the five adjuster questions twice over, ten calls in a row, each with the same
manual in front.

In [8]:
live_calls = [measure_question(question) for question in ADJUSTER_QUESTIONS * 2]

print(f"{MODEL}, the same manual in front of {len(live_calls)} questions:")
print_measurements(live_calls)
hits = [n for n, call in enumerate(live_calls, start=1) if call["cached"]]
print(f"\ncalls served partly from the cache: {hits or 'none'}")

google/gemini-2.5-flash-lite, the same manual in front of 10 questions:
call  1: cached     0 of 11104  0.48s  list $0.001116  billed $0.001116  host Google
call  2: cached     0 of 11104  0.51s  list $0.001115  billed $0.001115  host Google
call  3: cached     0 of 11108  0.49s  list $0.001118  billed $0.001118  host Google
call  4: cached     0 of 11106  0.91s  list $0.001117  billed $0.001117  host Google
call  5: cached     0 of 11104  0.51s  list $0.001115  billed $0.001115  host Google
call  6: cached     0 of 11104  0.61s  list $0.001114  billed $0.001114  host Google
call  7: cached     0 of 11104  0.58s  list $0.001115  billed $0.001115  host Google
call  8: cached     0 of 11108  0.47s  list $0.001118  billed $0.001118  host Google
call  9: cached     0 of 11106  0.50s  list $0.001117  billed $0.001117  host Google
call 10: cached 10227 of 11104  0.49s  list $0.001114  billed $0.000194  host Google

calls served partly from the cache: [10]


The first nine calls report zero cached tokens, on a manual that did not change by one character.
Call 10 is the first one served from the cache: 10227 of its 11104 prompt tokens were reused, and
it was billed $0.000194 against a list price of $0.001114. A spot check that stopped after a few
calls would have reported that this model never caches, so a saving goes into a budget only after
`cached_tokens` has been read across many calls.

## Step 4: Show that caching belongs to the model

The same code can report very different caching on a different model, because caching is something
the provider offers per model and not something the request asks for. The next cell sends the same
manual and the same five questions to the repository's small model.

In [9]:
SMALL_MODEL = model_for("small")
small_calls = [measure_question(question, SMALL_MODEL) for question in ADJUSTER_QUESTIONS]

print(f"{SMALL_MODEL}, the same manual in front of the same five questions:")
print_measurements(small_calls)

mistralai/mistral-nemo, the same manual in front of the same five questions:
call  1: cached     0 of 11709  2.25s  list $0.000223  billed $0.000211  host DekaLLM
call  2: cached     0 of 11709  0.90s  list $0.000223  billed $0.000211  host DekaLLM
call  3: cached     0 of 11713  3.63s  list $0.000224  billed $0.000212  host DekaLLM
call  4: cached     0 of 11712  1.61s  list $0.000223  billed $0.000211  host DekaLLM
call  5: cached 11696 of 11708  0.83s  list $0.000223  billed $0.000342  host Io Net


`summarise_caching` adds the calls up, so the two models can be compared on one line each.

In [10]:
def summarise_caching(model, calls):
    """Share of prompt tokens served from a cache, and the bill against the list price."""
    prompt = sum(call["prompt"] for call in calls)
    cached = sum(call["cached"] or 0 for call in calls)
    listed = sum(call["list_usd"] for call in calls)
    billed = sum(call["billed_usd"] or call["list_usd"] for call in calls)
    reported = all(call["cached"] is not None for call in calls)
    print(f"{model:30} cached {cached / prompt:4.0%} of prompt tokens, list ${listed:.6f}, "
          f"billed ${billed:.6f}, reported on every call: {reported}")


summarise_caching(MODEL, live_calls)
summarise_caching(SMALL_MODEL, small_calls)

google/gemini-2.5-flash-lite   cached   9% of prompt tokens, list $0.011160, billed $0.010239, reported on every call: True
mistralai/mistral-nemo         cached  20% of prompt tokens, list $0.001116, billed $0.001188, reported on every call: True


Four of the five calls to the small model cached nothing, and all four were served by the same
host. The fifth went to a different host, which reused 11696 of its 11708 prompt tokens and still
billed $0.000342, more than the list price of $0.000223 for that call. Across the five calls the
small model was billed more than its list price, even though a fifth of its prompt tokens came from
a cache.

That is what it means for caching to be a property of the model. The model you call, and the host
that serves it, decide whether a cache exists and what a cached token costs, and nothing in this
request changes either. So the numbers to budget with are `cached_tokens` and the billed `cost`,
read from real responses. The cache details are also optional in the response format, so a model
may leave them out altogether, which is why `read_cached_tokens` returns `None` rather than zero.

## Step 5: Measure warm latency instead of the first call

A latency figure taken from one call, usually the first, says little about what an adjuster will
see later, so we compare it with the calls that came after. **Cold latency** is the time of a call
made before the connection and the provider's cache are ready, and **warm latency** is the time
once they are, which is what every later question gets.

![Measure warm latency instead of the first call](images/claims-lanes-step-2.svg)

In [11]:
def summarise_latency(model, first_seconds, later_calls):
    """Compare the first call with the median and slowest of the warm calls after it."""
    warm = [call["seconds"] for call in later_calls]
    print(f"{model:30} first call {first_seconds:5.2f}s   warm median "
          f"{statistics.median(warm):5.2f}s   warm slowest {max(warm):5.2f}s")


summarise_latency(MODEL, FIRST_CALL_SECONDS, live_calls)
summarise_latency(SMALL_MODEL, small_calls[0]["seconds"], small_calls[1:])

google/gemini-2.5-flash-lite   first call  0.56s   warm median  0.50s   warm slowest  0.91s
mistralai/mistral-nemo         first call  2.25s   warm median  1.26s   warm slowest  3.63s


On the default model the first call took about 0.55 seconds against a warm median of about 0.5, so
here the first call happened to be close. The small model's first call took about 2.3 seconds
against a warm median of about 1.3, almost twice as long, and its slowest warm call took about 3.6
seconds. One call is a
sample of one, cold or warm, so a latency budget is set from the median and the slowest of many
warm calls. The live lane below uses the default model, whose warm calls took about half a second.

## Step 6: Answer a claim that needs three lookups

A real question from an adjuster usually needs more than one answer, so the desk makes three calls
for one claim: which rule applies, what evidence is missing, and whether to escalate. The adjuster
is waiting on all three, and the desk promises an answer within `LIVE_BUDGET_SECONDS`, one second,
which a single warm call fits with room to spare.

In [12]:
LIVE_CLAIM = ("CLM-5521: a storm blew tiles off the roof and rain came into the bedroom. "
              "Repair quote 3200 EUR, photos attached, reported the same day.")
LOOKUPS = [f"{LIVE_CLAIM} Which rule matches this claim, and what excess applies?",
           f"{LIVE_CLAIM} Which evidence is still missing?",
           f"{LIVE_CLAIM} Must this claim be escalated, and why?"]
LIVE_BUDGET_SECONDS = 1.0


def time_in_sequence(call, items):
    """Make the calls one after another. Return every result and the wall clock."""
    started = time.monotonic()
    results = [call(item) for item in items]
    return results, time.monotonic() - started

The first version makes the three calls one after another, which is how most code is first
written.

In [13]:
results, serial_wall = time_in_sequence(ask_claims_desk, LOOKUPS)
serial_seconds = [seconds for _, seconds in results]

for response, seconds in results:
    print(f"{seconds:5.2f}s  {response.choices[0].message.content.strip()[:80]}")
print(f"\nsum of the calls : {sum(serial_seconds):.2f}s")
print(f"wall clock       : {serial_wall:.2f}s")
print(f"within the {LIVE_BUDGET_SECONDS}s budget: {serial_wall <= LIVE_BUDGET_SECONDS}")

 0.55s  RULE-002: peril storm, excess 200 EUR, fast track under 2000 EUR, needs a police
 0.64s  RULE-002: peril storm, excess 200 EUR, fast track under 2000 EUR, needs a police
 0.59s  This claim needs to be escalated because the repair quote of 3200 EUR exceeds th

sum of the calls : 1.78s
wall clock       : 1.78s
within the 1.0s budget: False


Each call took well under a second, yet the adjuster waited about 1.8 seconds and the promise was
broken. Calls made one after another add their times together, so every lookup added to this list
adds its own time to the wait.

## Step 7: Run the three lookups in parallel

The three lookups do not depend on each other, so the desk can send them at the same time and wait
for all of them together. The **latency shape** is whether calls wait in a queue, which adds their
times up, or overlap, which makes the wall clock the time of the slowest call.

![Run the three lookups in parallel](images/claims-lanes-step-3.svg)

In [14]:
def time_in_parallel(call, items):
    """Make the calls at the same time. The wall clock is set by the slowest one."""
    started = time.monotonic()
    with ThreadPoolExecutor(max_workers=len(items)) as pool:
        results = list(pool.map(call, items))
    return results, time.monotonic() - started


results, parallel_wall = time_in_parallel(ask_claims_desk, LOOKUPS)
parallel_seconds = [seconds for _, seconds in results]

print(f"each call        : {[round(seconds, 2) for seconds in parallel_seconds]}")
print(f"sum of the calls : {sum(parallel_seconds):.2f}s")
print(f"slowest call     : {max(parallel_seconds):.2f}s")
print(f"wall clock       : {parallel_wall:.2f}s")
print(f"within the {LIVE_BUDGET_SECONDS}s budget: {parallel_wall <= LIVE_BUDGET_SECONDS}")

each call        : [0.48, 0.72, 0.6]
sum of the calls : 1.80s
slowest call     : 0.72s
wall clock       : 0.72s
within the 1.0s budget: True


The same three calls, with the same tokens and the same bill, now finish in about 0.7 seconds,
inside the budget. The call times still add up to about 1.8 seconds, but the wall clock follows the
slowest call, and the thread pool adds almost nothing on top of it. So the way
to speed up a parallel answer is to speed up its slowest call, and tuning the faster ones changes
nothing the adjuster sees.

## Step 8: Score the overnight backlog on the live lane

The overnight backlog is the other job the desk does, and the easy way to build it is to reuse the
live lane: the fast model, one claim per call, with the manual in front of each. Nobody is waiting
for these scores, so the only number that matters here is what they cost.

In [15]:
SCORING_REQUEST = "Score this claim for fraud risk from 1 (low) to 5 (high). Reply with the number only."


def score_claim_live(claim):
    """The live lane: the fast model, one claim per call."""
    return client.chat.completions.create(
        model=MODEL, max_tokens=10,
        messages=build_messages(f"{SCORING_REQUEST}\n{claim['claim_id']}: {claim['text']}"))


def total_cost(responses):
    """List price and billed dollars, summed over many calls."""
    prices = [price_the_call(response) for response in responses]
    return sum(p[0] for p in prices), sum(p[1] if p[1] is not None else p[0] for p in prices)

The next cell scores all twelve claims in the backlog, one call each, and adds up the bill.

In [16]:
live_lane_responses = [score_claim_live(claim) for claim in OVERNIGHT_BACKLOG]
live_list_usd, live_billed_usd = total_cost(live_lane_responses)
LIVE_USD_PER_CLAIM = live_billed_usd / len(OVERNIGHT_BACKLOG)

cached_calls = sum(1 for r in live_lane_responses if read_cached_tokens(r.usage))

print(f"calls          : {len(live_lane_responses)}, {cached_calls} of them served from the cache")
print(f"prompt tokens  : {sum(r.usage.prompt_tokens for r in live_lane_responses)}")
print(f"list price     : ${live_list_usd:.6f}")
print(f"billed         : ${live_billed_usd:.6f}, or ${LIVE_USD_PER_CLAIM:.7f} per claim")

calls          : 12, 2 of them served from the cache
prompt tokens  : 133660
list price     : $0.013374
billed         : $0.011534, or $0.0009611 per claim


Twelve claims were billed $0.011534, or $0.0009611 per claim, and only 2 of the 12 calls were
served from the cache. Almost all of that money pays for the manual, which went out twelve times,
once in front of each claim. The fast model's speed bought nothing here, because nobody was waiting
for these scores.

## Step 9: Build a batch lane for work that can wait

A **batch lane** is a separate path for work that can wait, built for price rather than speed. It
sends many claims in one request, so the manual is paid for once per group rather than once per
claim, and it uses the cheapest model, because a slow answer costs nothing when nobody is waiting.

![Build a batch lane for work that can wait](images/claims-lanes-step-4.svg)

In [17]:
BATCH_MODEL = SMALL_MODEL
BATCH_SIZE = 6
NIGHTLY_CLAIMS = 90_000   # claims waiting to be scored each night at this desk
BATCH_REQUEST = ("Score every claim below for fraud risk from 1 (low) to 5 (high). Reply with one "
                 "JSON object that maps each claim id to its score, and nothing else.")

for model in (MODEL, BATCH_MODEL):
    print(f"{model:30} list price per prompt token ${PRICES[model]['prompt_usd_per_token']}")

google/gemini-2.5-flash-lite   list price per prompt token $0.0000001
mistralai/mistral-nemo         list price per prompt token $0.000000019


`score_claims_in_batch` sends one group of claims. A reply cut off by its output budget has
**finish_reason**, the field on a response that says why the model stopped talking, set to
`length`, and a cut-off list of scores is refused rather than half used.

In [18]:
def score_claims_in_batch(claims):
    """The batch lane: the cheapest model, many claims in one request."""
    lines = "\n".join(f"{claim['claim_id']}: {claim['text']}" for claim in claims)
    response = client.chat.completions.create(
        model=BATCH_MODEL, max_tokens=300, messages=build_messages(f"{BATCH_REQUEST}\n{lines}"))
    if response.choices[0].finish_reason == "length":
        raise RuntimeError(f"a batch of {len(claims)} claims was cut off, so send fewer")
    return response


def read_batch_scores(response):
    """Parse the reply as JSON, allowing for a code fence around it."""
    text = response.choices[0].message.content.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)


def find_missing_claims(claims, scores):
    """Claims the reply did not score. They go back in the queue."""
    return [claim for claim in claims if claim["claim_id"] not in scores]

The batch lane works through a queue. Any claim a reply leaves out goes back in the queue, and a
round limit stops a claim that is never scored from looping forever.

In [19]:
MAX_BATCH_ROUNDS = 4
queue, scores, batch_responses = list(OVERNIGHT_BACKLOG), {}, []

for round_number in range(1, MAX_BATCH_ROUNDS + 1):
    if not queue:
        break
    group, queue = queue[:BATCH_SIZE], queue[BATCH_SIZE:]
    response = score_claims_in_batch(group)
    batch_responses.append(response)
    scores.update(read_batch_scores(response))
    missing = find_missing_claims(group, scores)
    queue += missing
    print(f"round {round_number}: sent {len(group)} claims, {len(missing)} missing from the reply")

batch_list_usd, batch_billed_usd = total_cost(batch_responses)
BATCH_USD_PER_CLAIM = batch_billed_usd / len(OVERNIGHT_BACKLOG)
print(f"\nscored {len(scores)} of {len(OVERNIGHT_BACKLOG)} claims in {len(batch_responses)} requests")
print(f"billed ${batch_billed_usd:.6f}, or ${BATCH_USD_PER_CLAIM:.7f} per claim")
print(f"the live lane cost {LIVE_USD_PER_CLAIM / BATCH_USD_PER_CLAIM:.0f} times as much per claim")
print(f"{NIGHTLY_CLAIMS} claims a night: live lane ${LIVE_USD_PER_CLAIM * NIGHTLY_CLAIMS:.2f}, "
      f"batch lane ${BATCH_USD_PER_CLAIM * NIGHTLY_CLAIMS:.2f}")

round 1: sent 6 claims, 0 missing from the reply


round 2: sent 6 claims, 0 missing from the reply

scored 12 of 12 claims in 2 requests
billed $0.000575, or $0.0000479 per claim
the live lane cost 20 times as much per claim
90000 claims a night: live lane $86.50, batch lane $4.32


Both requests came back with a score for every claim, so nothing went back in the queue in this
run. The check stays anyway, because nothing in the request forces the model to return every claim
id. The batch lane was billed $0.0000479 per claim against $0.0009611 on the live lane, twenty
times less, which is $4.32 a night instead of $86.50 for a backlog of 90000 claims.

Some providers also sell a separate batch endpoint that returns results hours later at a lower
price. This repository's client only calls the chat endpoint, so the batch lane here is built in
your own code, and this course does not measure a provider's batch price.

## Step 10: Route every job to the lane that suits it

The desk now has two lanes, and the last piece is the rule that chooses between them. The question
it asks is whether a person is waiting, because that decides whether the job is paid for in seconds
or in dollars.

![Route every job to the lane that suits it](images/claims-lanes-step-5.svg)

In [20]:
LANES = {
    "live": f"{MODEL}, lookups in parallel, answer within {LIVE_BUDGET_SECONDS}s",
    "batch": f"{BATCH_MODEL}, {BATCH_SIZE} claims per request, ready by the morning",
}


def choose_lane(job):
    """A job with an adjuster waiting goes live. Everything else waits for the batch lane."""
    return "live" if job.get("adjuster_waiting") else "batch"


jobs = ([{"work": question, "adjuster_waiting": True} for question in ADJUSTER_QUESTIONS[:2]]
        + [{"work": claim["claim_id"], "adjuster_waiting": False} for claim in OVERNIGHT_BACKLOG[:3]])
for job in jobs:
    lane = choose_lane(job)
    print(f"{job['work']:40} -> {lane:5}  {LANES[lane]}")

What excess applies under RULE-014?      -> live   google/gemini-2.5-flash-lite, lookups in parallel, answer within 1.0s
What evidence does RULE-122 need?        -> live   google/gemini-2.5-flash-lite, lookups in parallel, answer within 1.0s
CLM-4100                                 -> batch  mistralai/mistral-nemo, 6 claims per request, ready by the morning
CLM-4101                                 -> batch  mistralai/mistral-nemo, 6 claims per request, ready by the morning
CLM-4102                                 -> batch  mistralai/mistral-nemo, 6 claims per request, ready by the morning


The two lanes pay for the same claims desk in different currencies. Every number in this table was
printed by a cell above.

| Lane | Model | Shape | What it measured |
|---|---|---|---|
| Live | the default model | three lookups in parallel | about 0.7 seconds, against about 1.8 seconds in a row |
| Batch | the small model | six claims per request | $0.0000479 per claim, against $0.0009611 live |

## Step 11: Test the lanes without calling the model

Each rule above gets a test that runs in well under a second with no API key, so it can run on
every commit. If someone puts a timestamp in front of the manual, turns the lookups back into a
loop, or drops the check for missing claims, one of these tests fails.

In [21]:
def test_absent_cache_report_reads_as_none():
    assert read_cached_tokens(SimpleNamespace(prompt_tokens=10)) is None
    zero = SimpleNamespace(prompt_tokens_details=SimpleNamespace(cached_tokens=0))
    assert read_cached_tokens(zero) == 0


def test_manual_prefix_never_changes():
    first, second = build_messages("question one"), build_messages("question two")
    assert first[0] == second[0], "the manual changed between questions, so no cache can reuse it"


def test_lookups_overlap():
    def wait_a_moment(_):
        time.sleep(0.2)
    _, wall = time_in_parallel(wait_a_moment, range(3))
    assert wall < 0.4, f"three calls of 0.2s took {wall:.2f}s, so they ran in a queue"

The last two tests cover the batch lane's queue and the rule that picks a lane.

In [22]:
def test_missing_claims_go_back_in_the_queue():
    claims = [{"claim_id": "CLM-1"}, {"claim_id": "CLM-2"}]
    assert find_missing_claims(claims, {"CLM-1": 2}) == [{"claim_id": "CLM-2"}]


def test_waiting_adjuster_goes_live():
    assert choose_lane({"adjuster_waiting": True}) == "live"
    assert choose_lane({"adjuster_waiting": False}) == "batch"


for test in (test_absent_cache_report_reads_as_none, test_manual_prefix_never_changes,
             test_lookups_overlap, test_missing_claims_go_back_in_the_queue,
             test_waiting_adjuster_goes_live):
    test()
    print(f"passed: {test.__name__}")

passed: test_absent_cache_report_reads_as_none
passed: test_manual_prefix_never_changes


passed: test_lookups_overlap
passed: test_missing_claims_go_back_in_the_queue
passed: test_waiting_adjuster_goes_live


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Usage block** | `response.usage` | Counts the prompt and completion tokens, and says what the call was billed |
| **Prompt caching** | `read_cached_tokens` | Reads `cached_tokens`, and returns `None` when a model does not report it |
| **Stable prefix** | `build_messages` | Puts the same manual first in every request |
| **Warm latency** | `summarise_latency` | Times the calls after the first, not the first |
| **Latency shape** | `time_in_sequence` and `time_in_parallel` | A queue adds the call times, overlapping calls wait only for the slowest |
| **Batch lane** | `score_claims_in_batch` | Scores many claims in one request on the cheapest model, for work that can wait |
| **Missing claims** | `find_missing_claims` | Puts any claim a reply left out back in the queue |
| **Lane choice** | `choose_lane` | Sends a job live only when an adjuster is waiting |